<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/14-generative-autoregressive-models.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **生成建模基础与自回归模型** {#generative-modeling-foundations-autoregressive-models}

生成建模要求系统表示观测数据如何分布，并创建符合某个条件、上下文或已学习数据分布的新观测。自回归模型通过规定一个顺序、每次预测一个变量，使这一问题变得可处理。同一种数学形式可以生成文本 token、图像像素、音频采样点、动作、分子符号或结构化记录。

本章使用 scikit-learn 收录的 [UCI Optical Recognition of Handwritten Digits 数据集](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B)，DOI 为 [10.24432/C50P49](https://doi.org/10.24432/C50P49)，采用 CC BY 4.0 许可。每张真实的 8×8 图像被量化为四个灰度级，再按光栅顺序展平为 64 个离散 token。我们只训练一次类别条件循环解码器，随后在似然、暴露偏差、解码、采样控制和评估实验中复用它。

![由 UCI 数字图像量化得到的四级自回归 token 序列。](assets/dl14-quantized-digits.svg){fig-align="center" width="76%" fig-alt="十张量化手写数字图像，每张图像都与其光栅顺序像素 token 序列的开头配对。"}

*基于本章数据集生成的原创可视化；量化规则和光栅顺序与可执行实验一致。*

四级表示是有意采用的简化方案。它让模型可以在 CPU 上运行，并使每一步概率计算都能被观察，但不能据此得出高分辨率图像生成方面的结论。生成样本用于验证机制，而不是充当视觉生成基准。

<details>
<summary><strong>PyTorch：建立离散图像 token 实验</strong></summary>

```python
import copy
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1414):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))
train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1414, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1414,
    stratify=digits.target[holdout_idx],
)
train_images, train_labels = all_images[train_idx], all_labels[train_idx]
val_images, val_labels = all_images[val_idx], all_labels[val_idx]
test_images, test_labels = all_images[test_idx], all_labels[test_idx]

num_levels = 4
bos_token = num_levels


def quantize(images):
    return torch.round(images * (num_levels - 1)).long().flatten(1)


def dequantize(tokens):
    return tokens.float().view(-1, 1, 8, 8) / (num_levels - 1)


def decoder_inputs(tokens):
    bos = torch.full((len(tokens), 1), bos_token, dtype=torch.long)
    return torch.cat([bos, tokens[:, :-1]], dim=1)


train_tokens, val_tokens, test_tokens = quantize(train_images), quantize(val_images), quantize(test_images)
train_decoder_inputs = decoder_inputs(train_tokens)
val_decoder_inputs = decoder_inputs(val_tokens)
test_decoder_inputs = decoder_inputs(test_tokens)

assert all_images.shape == (1797, 1, 8, 8)
assert train_tokens.shape == (1257, 64)
assert train_decoder_inputs[:, 0].eq(bos_token).all()
assert train_tokens.min() == 0 and train_tokens.max() == 3
print({
    "split": (len(train_idx), len(val_idx), len(test_idx)),
    "sequence length": train_tokens.shape[1],
    "pixel vocabulary": list(range(num_levels)),
    "BOS token": bos_token,
    "token frequencies": torch.bincount(train_tokens.flatten(), minlength=num_levels).tolist(),
})
```

</details>

数据划分发生在量化之前。任何生成图像、验证图像或测试图像都不会进入训练集。量化是不含拟合统计量的固定变换，因此不会泄漏留出数据的信息。


### **生成模型学到的是什么？** {#what-does-a-generative-model-learn}

无条件模型近似 $p_{\text{data}}(x)$；条件模型近似 $p_{\text{data}}(x\mid c)$，其中 $c$ 可以是类别、提示词、图像、说话人、控制信号或先前状态。学习这一分布可以支持多种操作：

- 采样新的 $x$；
- 在似然可用时为观测打分或比较观测；
- 通过条件推断补全缺失部分；
- 利用预测概率压缩数据；
- 学习可用于下游任务的表示。

这些操作并不等价。模型可能给出有用的似然，却因解码器不合适而生成较差样本；隐式模型可能在不暴露 $p(x)$ 的情况下生成逼真样本；条件生成器也可能忽略 $c$，同时仍然匹配边缘分布 $p(x)$。训练目标、模型族与推断算法共同决定系统真正具备哪些能力。

本章希望学习的分布为

$$
p_{\theta}(x\mid y),\qquad x\in\{0,1,2,3\}^{64},\quad y\in\{0,\ldots,9\}.
$$

$x$ 是量化后的图像序列，$y$ 是目标数字类别。有效模型不应只是复制训练样本，而应当把概率分配给同一类别中合理的多种写法。模型还必须表达不确定性：在背景像素处，灰度级 0 可能占主导；在笔画边界附近，多个灰度级都可能合理。

“学习数据分布”始终是相对于特定数据集和表示而言的。UCI 数据没有覆盖所有书写风格，量化会丢失强度细节，光栅顺序则偏向从左到右、从上到下的依赖关系。因此，模型学到的是这条处理流水线所定义的分布，而不是抽象且普适的数字分布。


### **生成式建模与判别式建模** {#generative-vs-discriminative-modeling}

判别式分类器直接建模 $p(y\mid x)$ 或决策边界。生成式分类器建模 $p(x,y)=p(x\mid y)p(y)$，再通过

$$
p(y\mid x)=\frac{p(x\mid y)p(y)}{\sum_{y'}p(x\mid y')p(y')}
$$

得到后验概率。生成式路线能够采样 $x$、处理某些形式的缺失数据，并纳入先验；但它还必须描述与分类可能无关的输入细节，因此错误的密度假设会降低预测准确率。判别式路线可以把容量集中在决策边界，却不会自动定义如何生成输入。

![判别式模型学习条件标签边界，生成式模型则联合表示观测与标签。](assets/dl14-generative-discriminative.svg){fig-align="center" width="76%" fig-alt="左图展示从观测 x 到标签 y 的判别映射，右图展示先采样类别和观测的生成式联合模型。"}

*原创对比图。*

下面的实验比较 MLP 分类器与类别条件的朴素离散像素模型。后者拥有精确似然，但假设给定类别后各像素条件独立。

<details>
<summary><strong>PyTorch：比较判别式分类器与类别条件生成式分类器</strong></summary>

```python
class DigitDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU())
        self.head = nn.Linear(64, 10)

    def forward(self, images, return_features=False):
        features = self.encoder(images)
        logits = self.head(features)
        return (logits, features) if return_features else logits


seed_everything(1420)
discriminative_model = DigitDiscriminator()
optimizer = torch.optim.AdamW(discriminative_model.parameters(), lr=3e-3, weight_decay=1e-4)
for _ in range(55):
    optimizer.zero_grad()
    loss = F.cross_entropy(discriminative_model(train_images), train_labels)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    discriminative_accuracy = float((
        discriminative_model(test_images).argmax(1) == test_labels
    ).float().mean())


def fit_naive_categorical(tokens, labels, alpha=1.0):
    counts = torch.full((10, 64, num_levels), alpha)
    for label in range(10):
        class_tokens = tokens[labels == label]
        for level in range(num_levels):
            counts[label, :, level] += (class_tokens == level).sum(dim=0)
    return (counts / counts.sum(dim=-1, keepdim=True)).log()


naive_log_prob = fit_naive_categorical(train_tokens, train_labels, alpha=1.0)
class_log_prior = torch.bincount(train_labels, minlength=10).float().log()
class_log_prior -= torch.logsumexp(class_log_prior, dim=0)


def naive_class_scores(tokens):
    scores = []
    positions = torch.arange(64).unsqueeze(0)
    for label in range(10):
        token_log_prob = naive_log_prob[label][positions, tokens]
        scores.append(token_log_prob.sum(dim=1) + class_log_prior[label])
    return torch.stack(scores, dim=1)


with torch.no_grad():
    generative_accuracy = float((naive_class_scores(test_tokens).argmax(1) == test_labels).float().mean())


def sample_naive_class(label, count, seed=1420):
    generator = torch.Generator().manual_seed(seed + int(label))
    probabilities = naive_log_prob[label].exp()
    samples = torch.multinomial(probabilities, count, replacement=True, generator=generator).T
    return samples


naive_samples = torch.cat([sample_naive_class(label, 2) for label in range(10)], dim=0)
assert naive_samples.shape == (20, 64)
print({"discriminative test accuracy": round(discriminative_accuracy, 3),
       "generative Naive Bayes accuracy": round(generative_accuracy, 3),
       "generated token range": (int(naive_samples.min()), int(naive_samples.max()))})
```

</details>

该结果并不能证明判别学习总是更好。生成式基线故意采用较弱的独立性假设，而 MLP 能学习特征之间的交互。严格的科学比较需要控制参数量、分别调优两种方法、评估缺失数据任务，并同时量化分类性能和样本质量。


### **显式密度模型与隐式密度模型** {#explicit-implicit-density-models}

**显式密度模型**定义或近似 $p_{\theta}(x)$。自回归模型和归一化流提供可处理的似然；潜变量模型则经常优化一个下界，因为对潜变量 $z$ 做边缘化通常不可处理。**隐式模型**定义诸如 $x=G_{\theta}(z)$ 的采样过程，却不提供任意 $x$ 上可处理且归一化的密度。

![不同生成模型族在似然访问方式、推断与采样路径上存在差异。](assets/dl14-density-taxonomy.svg){fig-align="center" width="76%" fig-alt="显式模型包括可处理似然和近似下界模型，隐式模型则提供样本而不暴露逐点密度。"}

*原创分类图。潜变量、流、能量、GAN 与扩散模型将在第 15 和第 16 章展开。*

可访问似然有利于压缩和异常评分，但似然可能偏好与语义典型性不一致的低层统计特征。只能采样的模型可以采用灵活的生成器，但必须通过样本或判别器比较分布。两类模型都不保证采样迅速：显式自回归模型可能需要数千个串行步骤，而隐式前馈生成器可能只需一次前向传播。

下面的代码对比精确的朴素密度与“自助采样加扰动”的生成过程。后者刻意保持简单：它定义了一个生成程序，但在裁剪和随机扰动后，无法为任意输出给出唯一且可处理的概率。

<details>
<summary><strong>Python：比较逐点似然访问与仅采样访问</strong></summary>

```python
def naive_sequence_log_prob(tokens, labels):
    positions = torch.arange(64).unsqueeze(0)
    rows = []
    for row, label in enumerate(labels):
        rows.append(naive_log_prob[int(label)][positions[0], tokens[row]].sum())
    return torch.stack(rows)


def implicit_bootstrap_sample(count, seed=1430):
    generator = torch.Generator().manual_seed(seed)
    selected = torch.randint(len(train_images), (count,), generator=generator)
    noise = 0.08 * torch.randn((count, 1, 8, 8), generator=generator)
    return (train_images[selected] + noise).clamp(0.0, 1.0)


explicit_test_nll = float(-naive_sequence_log_prob(test_tokens, test_labels).mean())
implicit_samples = implicit_bootstrap_sample(40)
nearest_distance = torch.cdist(implicit_samples.flatten(1), train_images.flatten(1)).min(dim=1).values

assert math.isfinite(explicit_test_nll)
assert implicit_samples.shape == (40, 1, 8, 8)
print({"explicit mean test NLL": round(explicit_test_nll, 2),
       "implicit samples": len(implicit_samples),
       "implicit mean nearest-train distance": round(float(nearest_distance.mean()), 3),
       "implicit log_prob(x)": "not available"})
```

</details>

接近零的最近邻距离会暴露复制行为，但较大距离并不能证明质量良好。本例中的扰动可能产生新颖却不像数字的数组。生成评估必须同时衡量保真度与覆盖度，并且所用特征空间必须适合当前领域。


### **最大似然估计** {#maximum-likelihood-estimation}

最大似然估计（MLE）选择能让已观测训练样本概率最大的参数：

$$
\theta^{*}=\arg\max_{\theta}\sum_{n=1}^{N}\log p_{\theta}(x^{(n)}),
\qquad
\mathcal{L}_{\text{NLL}}=-\frac{1}{N}\sum_{n=1}^{N}\log p_{\theta}(x^{(n)}).
$$

对于离散的下一 token 分布，负对数似然（NLL）就是观测 token 与预测概率之间的交叉熵。在数据分布熵固定时，最小化期望 NLL 等价于最小化 $\mathrm{KL}(p_{\text{data}}\|p_{\theta})$。这一 KL 方向会强烈惩罚模型对已观测模式分配过低概率，因而倾向于覆盖数据模式；但有限数据与模型设定错误仍会影响结果。

朴素像素模型有闭式 MLE：对每个类别、位置和灰度级的计数进行归一化。若不做平滑，验证集中从未在训练中出现的事件会得到零概率，从而产生无穷大的 NLL。Dirichlet/Laplace 伪计数 $\alpha$ 用一定偏差换取有限且完整的支持集。

<details>
<summary><strong>Python：检查平滑强度与留出集负对数似然</strong></summary>

```python
def categorical_model_nll(alpha):
    log_probability = fit_naive_categorical(train_tokens, train_labels, alpha=alpha)
    positions = torch.arange(64).unsqueeze(0)
    row_nll = []
    for row, label in enumerate(val_labels):
        selected = log_probability[int(label)][positions[0], val_tokens[row]]
        row_nll.append(-selected.sum())
    return float(torch.stack(row_nll).mean())


smoothing_results = {alpha: categorical_model_nll(alpha) for alpha in (0.01, 0.1, 1.0, 5.0)}
uniform_nll = 64 * math.log(num_levels)
best_alpha = min(smoothing_results, key=smoothing_results.get)
assert all(math.isfinite(value) for value in smoothing_results.values())
print({"validation NLL by alpha": {k: round(v, 2) for k, v in smoothing_results.items()},
       "uniform-model NLL": round(uniform_nll, 2), "selected alpha": best_alpha})
```

</details>

似然必须用可比较的单位报告。对图像而言，每维比特数为 $\mathrm{NLL}/(D\log 2)$，其中 $D$ 是被建模的标量维数。若模型采用不同量化、去量化、tokenization 或预处理，直接比较可能无效。训练 NLL 本身不能证明泛化；必须同时考察留出集似然和样本行为。


### **自回归分解** {#autoregressive-factorization}

链式法则可以把任意有序联合分布分解为

$$
p(x_1,\ldots,x_T\mid c)=\prod_{t=1}^{T}p(x_t\mid x_{<t},c),
\qquad
\log p(x\mid c)=\sum_{t=1}^{T}\log p(x_t\mid x_{<t},c).
$$

这一恒等式本身是精确的；近似来自神经条件分布和所选择的顺序。顺序会改变哪些依赖更容易学习。图像的光栅顺序偏向前面行列；文本从左到右适合续写，却不便于双向补全；彩色图像还必须考虑通道顺序。

![一个 64-token 图像的似然被分解为有序条件概率。](assets/dl14-autoregressive-factorization.svg){fig-align="center" width="76%" fig-alt="像素 token x1 到 x64 构成一条链，每个条件概率只依赖更早的 token。"}

*基于 [PixelRNN](https://proceedings.mlr.press/v48/oord16.html) 中自回归图像形式绘制的原创链式法则图。*

训练时每个真实前缀都已知，因此可以在一个 batch 中累计全部 token 的损失。生成时必须先选择 $x_t$，才能得到 $x_{t+1}$ 所需的前缀。下面的循环模型学习 64 个量化像素上的类别条件分布。

<details>
<summary><strong>PyTorch：训练一个类别条件自回归像素解码器</strong></summary>

```python
class ConditionalPixelRNN(nn.Module):
    def __init__(self, embedding_dim=24, hidden_dim=72):
        super().__init__()
        self.token_embedding = nn.Embedding(num_levels + 1, embedding_dim)
        self.class_embedding = nn.Embedding(10, embedding_dim)
        self.rnn = nn.GRU(embedding_dim * 2, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, num_levels)

    def forward(self, input_tokens, labels):
        token_state = self.token_embedding(input_tokens)
        class_state = self.class_embedding(labels).unsqueeze(1).expand(-1, input_tokens.shape[1], -1)
        hidden, _ = self.rnn(torch.cat([token_state, class_state], dim=-1))
        return self.output(hidden)


def sequence_nll(model, inputs, targets, labels):
    model.eval()
    with torch.no_grad():
        logits = model(inputs, labels)
        per_token = F.cross_entropy(logits.transpose(1, 2), targets, reduction="none")
    return per_token.sum(dim=1)


seed_everything(1440)
autoregressive_model = ConditionalPixelRNN()
optimizer = torch.optim.AdamW(autoregressive_model.parameters(), lr=2e-3, weight_decay=1e-4)
loader = DataLoader(
    TensorDataset(train_decoder_inputs, train_tokens, train_labels),
    batch_size=160, shuffle=True, generator=torch.Generator().manual_seed(1440),
)
best_state, best_validation_nll = copy.deepcopy(autoregressive_model.state_dict()), float("inf")
for _ in range(42):
    autoregressive_model.train()
    for input_batch, target_batch, label_batch in loader:
        optimizer.zero_grad()
        logits = autoregressive_model(input_batch, label_batch)
        loss = F.cross_entropy(logits.transpose(1, 2), target_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(autoregressive_model.parameters(), 1.0)
        optimizer.step()
    validation_nll = float(sequence_nll(
        autoregressive_model, val_decoder_inputs, val_tokens, val_labels
    ).mean())
    if validation_nll < best_validation_nll:
        best_validation_nll = validation_nll
        best_state = copy.deepcopy(autoregressive_model.state_dict())
autoregressive_model.load_state_dict(best_state)

test_ar_nll = float(sequence_nll(
    autoregressive_model, test_decoder_inputs, test_tokens, test_labels
).mean())
assert test_ar_nll < uniform_nll
print({"best validation NLL": round(best_validation_nll, 2),
       "test autoregressive NLL": round(test_ar_nll, 2),
       "test bits/dimension": round(test_ar_nll / (64 * math.log(2)), 3)})
```

</details>

RNN 在不同位置共享参数，类别嵌入则提供条件 $c=y$。除光栅顺序外，它没有显式的二维归纳偏置。更好的似然并不保证样本在视觉上更受偏好；条件模型还可能利用类别不平衡或忽略条件，因此两项都必须单独测试。


### **Teacher Forcing 与暴露偏差** {#teacher-forcing-exposure-bias}

Teacher forcing 在训练时提供真实的前一个 token：

$$
\mathcal{L}_{\text{TF}}=-\sum_{t=1}^{T}\log p_{\theta}(x_t^{*}\mid x_{<t}^{*},c).
$$

推断时，模型却以自己采样得到的历史 $\hat x_{<t}$ 为条件。早期的一个错误 token 可能让模型进入训练期间很少见的前缀区域，之后的错误便可能累积。这种训练与推断上下文不匹配通常称为暴露偏差（exposure bias）。

![Teacher forcing 使用真实历史，自由运行生成则把模型输出反馈为后续上下文。](assets/dl14-teacher-forcing.svg){fig-align="center" width="74%" fig-alt="训练行使用真实的前一个 token，推断行使用采样 token，早期错误会改变后续上下文。"}

*参考 [Scheduled Sampling](https://papers.nips.cc/paper_files/paper/2015/hash/e995f98d56967d946471af29d7bf99f1-Abstract.html) 绘制的原创教学图。*

Scheduled sampling 会在训练期间偶尔用生成 token 替换真实前缀，但这会改变训练分布，并非普遍一致的修复方案。序列级目标、模仿学习、去噪、鲁棒条件建模和更好的模型校准都是其他路径。首要诊断应当是测量模型对前缀扰动的敏感性，而不是把所有差样本都归因于暴露偏差。

<details>
<summary><strong>PyTorch：测量 teacher-forced 准确率与前缀错误传播</strong></summary>

```python
autoregressive_model.eval()
with torch.no_grad():
    teacher_logits = autoregressive_model(test_decoder_inputs, test_labels)
    teacher_token_accuracy = float((teacher_logits.argmax(-1) == test_tokens).float().mean())

    clean_prefix = test_decoder_inputs[:64].clone()
    corrupted_prefix = clean_prefix.clone()
    # Position 9 is the input carrying the true token from target position 8.
    corrupted_prefix[:, 9] = (corrupted_prefix[:, 9] + 1) % num_levels
    clean_logits = autoregressive_model(clean_prefix, test_labels[:64])
    corrupted_logits = autoregressive_model(corrupted_prefix, test_labels[:64])
    clean_log_prob = clean_logits.log_softmax(-1)
    corrupt_log_prob = corrupted_logits.log_softmax(-1)
    propagation_kl = F.kl_div(
        corrupt_log_prob[:, 9:], clean_log_prob[:, 9:].exp(), reduction="none"
    ).sum(-1).mean(dim=0)

assert teacher_token_accuracy > 0.70
assert torch.all(propagation_kl >= -1e-6)
print({"teacher-forced token accuracy": round(teacher_token_accuracy, 3),
       "mean KL immediately after corruption": round(float(propagation_kl[0]), 4),
       "mean KL ten steps later": round(float(propagation_kl[min(10, len(propagation_kl)-1)]), 4)})
```

</details>

自由生成序列与参考序列逐像素比较的准确率并不是有效的主要指标，因为许多不同图像都可以表示同一数字。这里的 KL 诊断改为询问：一个被改变的历史是否会改变后续预测分布。在语言任务中，还必须把误差累积与表达歧义区分开；生成续写即使不同于单一参考答案，也可能仍然有效。


### **PixelCNN 与 WaveNet** {#pixelcnn-wavenet}

自回归卷积用因果感受野替代循环。PixelCNN 使用带掩码的二维卷积预测图像像素：第一层的 A 类掩码排除当前像素；后续 B 类掩码可以包含当前位置的隐藏表示，但仍排除未来原始像素。[PixelRNN/PixelCNN](https://proceedings.mlr.press/v48/oord16.html) 表明，离散像素似然能够建模复杂的图像依赖。

WaveNet 把因果一维卷积应用于原始音频。膨胀率 $1,2,4,\ldots$ 会让感受野随深度指数扩展。对于卷积核大小 $k$ 和膨胀率序列 $d_l$，感受野为

$$
R=1+(k-1)\sum_l d_l.
$$

门控激活、残差连接、跳跃连接与条件信息帮助模型处理长波形。[WaveNet](https://arxiv.org/abs/1609.03499) 至今仍是采样级自回归建模的经典示例。

![PixelCNN 遮蔽未来光栅位置，WaveNet 的膨胀卷积扩展因果时间感受野。](assets/dl14-causal-convolutions.svg){fig-align="center" width="76%" fig-alt="光栅掩码只暴露过去像素，膨胀时间卷积树则能连接更远的早期采样点。"}

*基于 [PixelRNN](https://proceedings.mlr.press/v48/oord16.html) 与 [WaveNet](https://arxiv.org/abs/1609.03499) 绘制的原创结构图。*

<details>
<summary><strong>PyTorch：构造因果 PixelCNN 掩码与 WaveNet 感受野</strong></summary>

```python
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if mask_type not in {"A", "B"}:
            raise ValueError(mask_type)
        self.register_buffer("mask", torch.ones_like(self.weight))
        center_y, center_x = self.kernel_size[0] // 2, self.kernel_size[1] // 2
        self.mask[:, :, center_y + 1:, :] = 0
        self.mask[:, :, center_y, center_x + 1:] = 0
        if mask_type == "A":
            self.mask[:, :, center_y, center_x] = 0

    def forward(self, inputs):
        return F.conv2d(inputs, self.weight * self.mask, self.bias,
                        self.stride, self.padding, self.dilation, self.groups)


masked_a = MaskedConv2d("A", 1, 4, kernel_size=3, padding=1, bias=False)
masked_b = MaskedConv2d("B", 4, 4, kernel_size=3, padding=1, bias=False)
assert masked_a.mask[0, 0, 1, 1] == 0
assert masked_b.mask[0, 0, 1, 1] == 1
assert masked_a.mask[0, 0, 1, 2] == 0


def receptive_field(kernel_size, dilations):
    return 1 + (kernel_size - 1) * sum(dilations)


dilations = [1, 2, 4, 8, 16]
wave_receptive_field = receptive_field(kernel_size=2, dilations=dilations)
assert wave_receptive_field == 32
print({"PixelCNN A-mask active weights": int(masked_a.mask.sum()),
       "PixelCNN B-mask active weights": int(masked_b.mask.sum()),
       "WaveNet dilations": dilations, "receptive field": wave_receptive_field})
```

</details>

卷积可以在训练时并行处理各位置，但朴素的祖先采样仍然是串行的，因为每个新像素或音频采样都会改变下一步输入。缓存与专用内核可以减少重复计算；多尺度、子尺度或潜变量方法则以放弃部分精确顺序为代价加快生成。掩码正确性必须通过单元测试保证：任何未来像素都不能影响更早的输出。


### **自回归解码器模型** {#autoregressive-decoder-models}

仅解码器 Transformer 对先前 token 做嵌入、加入位置信息、应用因果自注意力和前馈模块，再预测下一 token 分布。因果掩码把未来位置的注意力 logit 设为 $-\infty$。与 RNN 不同，teacher-forced 训练能够并行处理所有位置；与掩码编码器不同，每个表示只能使用自己的前缀。

![Teacher-forced 解码器训练并行预测所有位置，推断则每次附加一个 token。](assets/dl14-decoder-training-inference.svg){fig-align="center" width="76%" fig-alt="训练面板同时处理全部被因果遮罩的位置，推断面板反复采样和追加 token，并复用键值缓存。"}

*基于 [Attention Is All You Need](https://arxiv.org/abs/1706.03762) 中因果解码器绘制的原创教学图。*

推断期间，键值缓存保存先前各层的 attention key 与 value。对于 batch 大小 $B$、头数 $H$、缓存长度 $T$ 和头维度 $d_h$，每层的 K 与 V 张量形状都约为 `[B, H, T, d_h]`。缓存内存随生成长度线性增长，而每个新 token 的注意力计算量仍随上下文长度增长。

下面的小型 Transformer 不会被训练来取代本章共享的 RNN。它只验证因果契约：改变未来输入 token 不得改变更早位置的 logits。

<details>
<summary><strong>PyTorch：实现并单元测试因果 Transformer 解码器</strong></summary>

```python
class TinyCausalDecoder(nn.Module):
    def __init__(self, width=32, heads=4, max_length=64):
        super().__init__()
        self.token_embedding = nn.Embedding(num_levels + 1, width)
        self.class_embedding = nn.Embedding(10, width)
        self.position = nn.Parameter(torch.zeros(1, max_length, width))
        layer = nn.TransformerEncoderLayer(width, heads, dim_feedforward=64,
                                           dropout=0.0, batch_first=True)
        self.blocks = nn.TransformerEncoder(layer, num_layers=2)
        self.output = nn.Linear(width, num_levels)

    def forward(self, input_tokens, labels):
        length = input_tokens.shape[1]
        hidden = self.token_embedding(input_tokens) + self.position[:, :length]
        hidden = hidden + self.class_embedding(labels).unsqueeze(1)
        causal_mask = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
        return self.output(self.blocks(hidden, mask=causal_mask))


seed_everything(1460)
causal_decoder = TinyCausalDecoder()
prefix_a = test_decoder_inputs[:3, :20].clone()
prefix_b = prefix_a.clone()
prefix_b[:, 12:] = torch.randint(0, num_levels, prefix_b[:, 12:].shape)
causal_decoder.eval()
with torch.no_grad():
    logits_a = causal_decoder(prefix_a, test_labels[:3])
    logits_b = causal_decoder(prefix_b, test_labels[:3])
    past_difference = float((logits_a[:, :12] - logits_b[:, :12]).abs().max())

parameter_count = sum(p.numel() for p in causal_decoder.parameters())
kv_elements_per_layer = 2 * 3 * 4 * 20 * (32 // 4)
assert past_difference < 1e-6
print({"decoder parameters": parameter_count,
       "future-to-past max difference": past_difference,
       "illustrative K/V elements per layer": kv_elements_per_layer})
```

</details>

架构修改、替换融合注意力、使用打包序列以及更新缓存逻辑后，都应重新运行因果测试。差一位错误可能在训练时泄漏目标 token，造成看似极佳的损失，却无法正常生成。训练输入必须右移，使位置 $t$ 的 logit 预测目标 $x_t$，而不是预测该位置已经可见的 token。


### **贪心、Beam 与基于采样的生成** {#greedy-beam-sampling-generation}

训练定义条件概率，解码则把这些概率转化为序列。

- **贪心解码**在每一步选择 $\arg\max_v p(v\mid x_{<t})$。它确定且便宜，但一次局部错误可能无法撤销。
- **Beam search**按累计对数概率保留前 $B$ 条部分序列。它近似寻找高概率序列，但候选被剪枝后不能保证全局最优。
- **采样**从条件分布中抽取 token。它能够表达不确定性与多样性，却也可能进入低质量的概率尾部。

![贪心、beam 与随机解码在同一概率树中探索不同路径。](assets/dl14-decoding-strategies.svg){fig-align="center" width="76%" fig-alt="概率树展示一条贪心路径、若干 beam 路径以及按概率加权抽取的采样路径。"}

*原创解码策略对比图。*

固定长度的像素序列不需要 EOS token 或长度归一化。文本 beam search 通常需要长度惩罚，因为负对数概率之和偏好短序列。覆盖惩罚与重复惩罚是任务相关启发式方法，而不是概率模型本身的性质。

<details>
<summary><strong>PyTorch：实现贪心、beam 与随机像素解码</strong></summary>

```python
def next_token_logits(model, generated, labels):
    bos = torch.full((len(labels), 1), bos_token, dtype=torch.long)
    model_input = bos if generated.shape[1] == 0 else torch.cat([bos, generated], dim=1)
    return model(model_input, labels)[:, -1]


@torch.no_grad()
def greedy_decode(model, labels, length=64):
    generated = torch.empty(len(labels), 0, dtype=torch.long)
    for _ in range(length):
        token = next_token_logits(model, generated, labels).argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, token], dim=1)
    return generated


@torch.no_grad()
def sample_decode(model, labels, temperature=1.0, length=64, seed=1470):
    generator = torch.Generator().manual_seed(seed)
    generated = torch.empty(len(labels), 0, dtype=torch.long)
    for _ in range(length):
        logits = next_token_logits(model, generated, labels) / temperature
        token = torch.multinomial(logits.softmax(-1), 1, generator=generator)
        generated = torch.cat([generated, token], dim=1)
    return generated


@torch.no_grad()
def beam_decode_one(model, label, beam_width=4, length=64):
    beams = [(torch.empty(0, dtype=torch.long), 0.0)]
    label_tensor = torch.tensor([label])
    for _ in range(length):
        candidates = []
        for sequence, score in beams:
            logits = next_token_logits(model, sequence.view(1, -1), label_tensor)
            values, indices = logits.log_softmax(-1).topk(beam_width, dim=-1)
            for value, token in zip(values[0], indices[0]):
                candidates.append((torch.cat([sequence, token.view(1)]), score + float(value)))
        beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]
    return beams[0]


autoregressive_model.eval()
class_labels = torch.arange(10)
greedy_samples = greedy_decode(autoregressive_model, class_labels)
stochastic_samples = sample_decode(autoregressive_model, class_labels, seed=1471)
beam_sample, beam_log_probability = beam_decode_one(autoregressive_model, label=3)

assert greedy_samples.shape == stochastic_samples.shape == (10, 64)
assert beam_sample.shape == (64,)
print({"greedy unique sequences": int(torch.unique(greedy_samples, dim=0).shape[0]),
       "sampled unique sequences": int(torch.unique(stochastic_samples, dim=0).shape[0]),
       "class-3 beam log probability": round(beam_log_probability, 2)})
```

</details>

对于可接受输出集合较窄且序列分数校准良好的任务，例如某些受约束的转导问题，beam search 较为合适。开放式生成在最大化似然的解码下经常变得重复或泛化。解码方式应依据应用效用选择，不能假设它会改善底层模型本身。


### **Temperature、Top-k 与 Nucleus Sampling** {#temperature-top-k-nucleus-sampling}

Temperature 对 logits $z_v$ 进行缩放：

$$
p_T(v)=\frac{\exp(z_v/T)}{\sum_j\exp(z_j/T)}.
$$

$T<1$ 会使分布更尖锐，$T>1$ 会使分布更平坦。Top-$k$ sampling 只保留概率最高的 $k$ 个 token。Nucleus 或 top-$p$ sampling 则保留累计概率达到 $p$ 的最小 token 集合。与固定 $k$ 不同，nucleus 集合大小会随模型不确定性变化。[Holtzman 等人](https://arxiv.org/abs/1904.09751) 为应对开放式神经文本生成中的退化问题提出了 nucleus sampling。

截断是一种解码启发式方法。它可以移除不可靠的尾部事件，但也会改变模型分布，并可能丢弃有效的稀有 token。较小的像素词表让 top-$k$/top-$p$ 的效果容易观察，却不能完全代表拥有数万 token 的语言词表。

<details>
<summary><strong>PyTorch：实现 temperature、top-k 与 top-p sampling</strong></summary>

```python
def filter_logits(logits, top_k=None, top_p=None):
    filtered = logits.clone()
    if top_k is not None:
        threshold = filtered.topk(min(top_k, filtered.shape[-1]), dim=-1).values[:, -1:]
        filtered = filtered.masked_fill(filtered < threshold, -float("inf"))
    if top_p is not None:
        sorted_logits, sorted_indices = filtered.sort(dim=-1, descending=True)
        cumulative = sorted_logits.softmax(-1).cumsum(-1)
        remove = cumulative - sorted_logits.softmax(-1) >= top_p
        sorted_logits = sorted_logits.masked_fill(remove, -float("inf"))
        filtered = torch.full_like(filtered, -float("inf")).scatter(1, sorted_indices, sorted_logits)
    return filtered


@torch.no_grad()
def controlled_sample(model, labels, temperature=1.0, top_k=None, top_p=None, seed=1480):
    generator = torch.Generator().manual_seed(seed)
    generated = torch.empty(len(labels), 0, dtype=torch.long)
    for _ in range(64):
        logits = next_token_logits(model, generated, labels) / temperature
        logits = filter_logits(logits, top_k=top_k, top_p=top_p)
        token = torch.multinomial(logits.softmax(-1), 1, generator=generator)
        generated = torch.cat([generated, token], dim=1)
    return generated


generation_labels = torch.arange(10).repeat_interleave(6)
sampling_configs = {
    "cold_T0.6": {"temperature": 0.6},
    "ancestral_T1.0": {"temperature": 1.0},
    "top_k_2": {"temperature": 1.0, "top_k": 2},
    "top_p_0.85": {"temperature": 1.0, "top_p": 0.85},
}
generated_sets = {
    name: controlled_sample(autoregressive_model, generation_labels, seed=1480 + offset, **config)
    for offset, (name, config) in enumerate(sampling_configs.items())
}

for name, samples in generated_sets.items():
    unique_ratio = torch.unique(samples, dim=0).shape[0] / len(samples)
    mean_pixel_variance = float(dequantize(samples).var(dim=0).mean())
    print({"strategy": name, "unique ratio": round(unique_ratio, 3),
           "mean pixel variance": round(mean_pixel_variance, 4)})

assert all(samples.shape == (60, 64) for samples in generated_sets.values())
```

</details>

低多样性可能表示确定且自信，也可能意味着模式坍塌；高多样性可能表示有用变化，也可能只是噪声。文本或代码任务中的重复率、熵、distinct-$n$、self-BLEU、通过率与人类偏好分别刻画不同方面。采样超参数是部署系统的一部分，必须与模型一起进行版本管理。


### **生成模型评估** {#evaluating-generative-models}

没有单一指标能够完整描述生成分布。评估应区分：

- **密度拟合：**当存在可比较似然时，使用留出集 NLL、perplexity 或 bits per dimension；
- **保真度/precision：**样本是否落在数据流形上；
- **覆盖度/recall：**重要数据模式是否得到表示；
- **条件一致性：**输出是否遵循标签、提示词或控制信号；
- **新颖性与记忆：**输出是否复制训练记录；
- **任务效用与人类判断：**样本是否满足真实使用目标。

[FID](https://proceedings.neurips.cc/paper/2017/hash/8a1d694707eb0fefe65871369074926d-Abstract.html) 比较高斯特征统计量，但结果依赖特征提取器和样本数量。[生成模型的 precision 与 recall](https://proceedings.neurips.cc/paper_files/paper/2018/hash/f7696a9b362ac5a51c3dc8f098b73923-Abstract.html) 将保真度与模式覆盖分开。两者都不应使用不适合当前领域的编码器计算后，再被描述为普适质量指标。

下面的紧凑审计仅把本章判别式数字分类器用作领域探针。它对每种采样策略测量类别条件准确率、分类器置信度、多样性和最近训练样本距离。这些指标可能彼此冲突。

<details>
<summary><strong>PyTorch：审计似然、条件保真度、多样性与复制行为</strong></summary>

```python
def generation_audit(tokens, intended_labels):
    images = dequantize(tokens)
    discriminative_model.eval()
    with torch.no_grad():
        logits, features = discriminative_model(images, return_features=True)
        probabilities = logits.softmax(-1)
        conditional_accuracy = float((probabilities.argmax(1) == intended_labels).float().mean())
        confidence = float(probabilities.max(dim=1).values.mean())
        nearest_train = torch.cdist(images.flatten(1), train_images.flatten(1)).min(dim=1).values
    return {
        "conditional accuracy": conditional_accuracy,
        "classifier confidence": confidence,
        "unique ratio": torch.unique(tokens, dim=0).shape[0] / len(tokens),
        "mean nearest-train distance": float(nearest_train.mean()),
        "pixel diversity": float(images.var(dim=0).mean()),
    }


audit_rows = {name: generation_audit(samples, generation_labels)
              for name, samples in generated_sets.items()}
for name, metrics in audit_rows.items():
    print({name: {key: round(value, 3) for key, value in metrics.items()}})

independent_test_nll = float(-naive_sequence_log_prob(test_tokens, test_labels).mean())
evaluation_summary = {
    "independent categorical bits/dim": independent_test_nll / (64 * math.log(2)),
    "autoregressive bits/dim": test_ar_nll / (64 * math.log(2)),
    "real-test classifier accuracy": discriminative_accuracy,
}
assert evaluation_summary["autoregressive bits/dim"] < 2.0
assert all(0.0 <= row["conditional accuracy"] <= 1.0 for row in audit_rows.values())
print({key: round(value, 3) for key, value in evaluation_summary.items()})
```

</details>

评估器只在真实数据上训练，因此其置信度面对生成伪影时可能不可靠。最近邻距离只能在原始像素几何中识别完全或近似复制。更稳健的研究还需要加入人工审查、多随机种子、类别/子群覆盖、训练数据提取测试，以及经过领域验证的编码器。报告分布指标时还应说明样本量与置信区间。


### **章节对比与总结** {#chapter-comparison-summary}

生成建模定义一个分布或采样过程；自回归建模通过把联合分布分解为有序的下一 token 条件分布，获得可处理似然。由此形成鲜明权衡：当所有真实位置已知时，teacher-forced 训练可以并行；祖先生成则天然需要串行执行。

| 决策 | 主要选择 | 收益 | 典型限制 |
|---|---|---|---|
| 建模目标 | $p(x)$、$p(x\mid c)$ 或 $p(x,y)$ | 采样、条件生成或生成式分类 | 数据集与表示共同定义目标 |
| 密度访问 | 显式或隐式 | 似然/压缩或灵活采样 | 可处理性不保证感知质量 |
| 顺序 | 光栅、时间、token、通道 | 精确的链式法则分解 | 顺序改变归纳偏置与延迟 |
| 架构 | RNN、掩码 CNN、因果 Transformer | 循环、局部性或全局上下文 | 串行生成与缓存成本 |
| 训练上下文 | teacher forcing 或混合/自馈历史 | 稳定 MLE 或对生成前缀的鲁棒性 | 暴露不匹配或有偏训练目标 |
| 解码 | 贪心、beam、祖先采样、截断采样 | 确定性、搜索或多样性 | 局部错误、退化或尾部噪声 |
| 评估 | NLL、保真度、覆盖度、条件、新颖性 | 互补证据 | 每个指标都有领域与估计器假设 |

UCI 实验在不更换数据的情况下串联了这些决策。类别条件朴素密度揭示了独立性假设的代价；循环解码器利用前缀依赖改善留出集似然；前缀扰动测量了错误传播；因果掩码通过单元测试；同一个解码器随后支持贪心、beam、temperature、top-$k$ 与 top-$p$ 生成，从而可以在相同模型下比较质量、多样性与复制指标。

实际工作流是：明确随机变量与条件契约；定义 tokenization 和顺序；验证因果移位与掩码；用可比较单位追踪留出集似然；区分模型与解码器；最后评估保真度、覆盖度、条件一致性、新颖性、鲁棒性和具体用途效用。样本是证据而非证明，一张看起来漂亮的样本网格无法确立分布质量。

第 15 章将放宽纯自回归构造，引入潜变量模型、变分推断、归一化流与能量模型。这些方法分别用潜在结构、可逆性或未归一化能量，交换精确的自回归分解。
